In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import*
df = spark.read.table("ecommerce_silver.silver.silver_amz_sales")

StatementMeta(, c087b22b-daa1-4920-aa8b-b56c953d9256, 3, Finished, Available, Finished, False)

In [3]:
# total revenue for the date 2022-06-26
#  SELECT date, status_group, SUM(net_amount) AS total_revenue
#  FROM ecommerce_gold.gold.gold_fact_sales
# WHERE date = '2022-06-26'
#   AND status_group = 'Delivered'
# GROUP BY date, status_group;

StatementMeta(, c087b22b-daa1-4920-aa8b-b56c953d9256, 5, Finished, Available, Finished, False)

In [4]:
# ============================================================
#  DIMENSION 1 — dim_product
#  Columns  : product_key, sku, asin, style, category, size
#  Why      : Answers all product/category KPIs
#             Category performance, Size analysis
# ============================================================
dim_product = df.select(
    "product_key",
    "sku",
    "asin",
    "style",
    "category",
    "size"
).distinct()

print("dim_product rows:", dim_product.count())

dim_product.write.format("delta").mode("overwrite").option("overwriteSchema","true" ).saveAsTable("ecommerce_gold.gold.gold_dim_product")
print("gold_dim_product saved")

StatementMeta(, c087b22b-daa1-4920-aa8b-b56c953d9256, 6, Finished, Available, Finished, False)

dim_product rows: 7190
gold_dim_product saved


In [5]:
# ============================================================
#  DIMENSION 2 — dim_location
#  Columns  : location_key, ship_city, ship_state,
#              ship_postal_code
#  Why      : Answers all geographic KPIs
#             City & State wise revenue,
#             Regional performance maps
# ============================================================
dim_location=df.select(
    "location_key",
     "ship_city",
    "ship_state",
    "ship_postal_code"
).distinct()

print("dim_location rows:", dim_location.count())
dim_location.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce_gold.gold.gold_dim_location")
print("gold_dim_location saved")

StatementMeta(, c087b22b-daa1-4920-aa8b-b56c953d9256, 7, Finished, Available, Finished, False)

dim_location rows: 13540
gold_dim_location saved


In [6]:
# ============================================================
#  DIMENSION 3 — dim_fulfillment
#  Columns  : fulfillment_key, fulfilment, ship_service_level
#  Why      : Answers operations/fulfillment KPIs
#             Amazon vs Merchant fulfillment
#             Standard vs Expedited shipping
# ============================================================
dim_fulfillment = df.select(
    "fulfillment_key",
    "fulfilment",
    "ship_service_level"
).distinct()

print("dim_fulfillment rows:", dim_fulfillment.count())

dim_fulfillment.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce_gold.gold.gold_dim_fulfillment")

print("gold_dim_fulfillment saved")

StatementMeta(, c087b22b-daa1-4920-aa8b-b56c953d9256, 8, Finished, Available, Finished, False)

dim_fulfillment rows: 3
gold_dim_fulfillment saved


In [7]:
# ============================================================
#  DIMENSION 4 — dim_date
#  Columns  : date, order_year, order_month, order_quarter,
#              order_weekday, month_name
#  Why      : Powers ALL time intelligence KPIs
#             Daily/Monthly/Quarterly revenue
#             YTD, MTD, Growth %, Peak Day
#             Weekday trends
# ============================================================
dim_date = df.select(
    "date",
    "order_year",
    "order_month",
    "order_quarter",
    "order_weekday",
    "month_name"
).distinct() \
 .orderBy("date")

print("dim_date rows:", dim_date.count())

dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce_gold.gold.gold_dim_date")

print("gold_dim_date saved")


StatementMeta(, c087b22b-daa1-4920-aa8b-b56c953d9256, 9, Finished, Available, Finished, False)

dim_date rows: 91
gold_dim_date saved


In [8]:
# ============================================================
#  DIMENSION 5 — dim_promotion
#  Columns  : promotion_type, has_promotion
#  Why      : Answers promotions dashboard KPIs
#             Promo vs Non-Promo revenue comparison
#             Which promo type drives most orders
# ============================================================
dim_promotion = df.select(
    "promotion_type",
    "has_promotion"
).distinct()

print("dim_promotion rows:", dim_promotion.count())

dim_promotion.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce_gold.gold.gold_dim_promotion")

print("gold_dim_promotion saved")



StatementMeta(, c087b22b-daa1-4920-aa8b-b56c953d9256, 10, Finished, Available, Finished, False)

dim_promotion rows: 5
gold_dim_promotion saved


In [9]:
# ============================================================
#  DIMENSION 6 — dim_customer
#  Columns  : customer_type
#  Why      : B2B vs B2C segmentation KPIs
#             Revenue split, AOV comparison
# ============================================================

dim_customer = df.select(
    "customer_type"
).distinct()

print("dim_customer rows:", dim_customer.count())

dim_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce_gold.gold.gold_dim_customer")

print("gold_dim_customer saved")


StatementMeta(, c087b22b-daa1-4920-aa8b-b56c953d9256, 11, Finished, Available, Finished, False)

dim_customer rows: 2
gold_dim_customer saved


In [10]:
# ============================================================
#  FACT TABLE — fact_sales
#  One row per order line item
#  Contains: all foreign keys + all measures
#  Why      : Central table that connects all dimensions
#             All KPI calculations happen on this table
# ============================================================
fact_sales = df.select(

    # ── Identifiers ──
    "order_id",
    "status",
    "status_group",
    "courier_status",
    "is_valid_revenue",

    # ── Foreign Keys ──
    "date",
    "product_key",
    "location_key",
    "fulfillment_key",
    "promotion_type",
    "customer_type",

    # ── Measures ──
    "amount",
    "net_amount",       # ← added
    "qty",
    "promo_count",
    "has_promotion",

    # ── Partition Columns ──
    "order_year",       # ← needed for partitionBy
    "order_month",      # ← needed for partitionBy
)

print("fact_sales rows:", fact_sales.count())

StatementMeta(, c087b22b-daa1-4920-aa8b-b56c953d9256, 12, Finished, Available, Finished, False)

fact_sales rows: 128748


In [11]:
# Verify row count matches to silver then insert
assert fact_sales.count() == df.count(), \
    "Row count mismatch — check fact_sales build!"

fact_sales.write.format("delta").mode("overwrite").partitionBy("order_year", "order_month") .option("overwriteSchema", "true").saveAsTable("ecommerce_gold.gold.gold_fact_sales")

print("gold_fact_sales saved")

StatementMeta(, c087b22b-daa1-4920-aa8b-b56c953d9256, 13, Finished, Available, Finished, False)

gold_fact_sales saved


In [14]:
from pyspark.sql.functions import (
    col, lit, md5, concat_ws,
    monotonically_increasing_id,
    countDistinct, sum as spark_sum,  # ← this line is missing
    avg, max as spark_max, min as spark_min
)
# ============================================================
#  FINAL VERIFICATION
# ============================================================

print("\n" + "="*60)
print("GOLD LAYER VERIFICATION")
print("="*60)

# Check all table row counts
tables = {
    "ecommerce_gold.gold.gold_fact_sales"       : "fact",
    "ecommerce_gold.gold.gold_dim_product"      : "dim",
    "ecommerce_gold.gold.gold_dim_location"     : "dim",
    "ecommerce_gold.gold.gold_dim_fulfillment"  : "dim",
    "ecommerce_gold.gold.gold_dim_date"         : "dim",
    "ecommerce_gold.gold.gold_dim_promotion"    : "dim",
    "ecommerce_gold.gold.gold_dim_customer"     : "dim",
}

for table, ttype in tables.items():
    count = spark.read.table(table).count()
    print(f"  {table:35s} → {count:>6} rows")

# Verify fact table KPI numbers
# print("\n--- Fact Table KPI Sanity Check ---")
# spark.read.table("ecommerce_gold.gold.gold_fact_sales").filter(col("is_valid_revenue") == True)\
#     .agg(
#         spark_sum("amount").alias("total_revenue"),
#         countDistinct("order_id").alias("total_orders"),
#         spark_sum("qty").alias("total_qty"),
#         (spark_sum("amount") /
#          countDistinct("order_id")).alias("aov")
#     ).show(truncate=False)


# Verify all joins work
print("\n--- Join Verification ---")
fact = spark.read.table("ecommerce_gold.gold.gold_fact_sales")
dim_p = spark.read.table("ecommerce_gold.gold.gold_dim_product")
dim_l = spark.read.table("ecommerce_gold.gold.gold_dim_location")
dim_f = spark.read.table("ecommerce_gold.gold.gold_dim_fulfillment")
dim_d = spark.read.table("ecommerce_gold.gold.gold_dim_date")

joined = fact \
    .join(dim_p, "product_key", "left") \
    .join(dim_l, "location_key", "left") \
    .join(dim_f, "fulfillment_key", "left") \
    .join(dim_d, "date", "left")

print("Joined rows:", joined.count())
print("Expected   :", fact.count())

if joined.count() == fact.count():
    print("All joins verified — no orphan keys!")
else:
    print("Row count mismatch — check surrogate keys!")


# print("\n" + "="*60)
# print("GOLD LAYER COMPLETE")
# print("="*60)
# print("""
# Tables created:
#   gold_fact_sales         → Main fact table (partitioned by date)
#   gold_dim_product        → Product / Category dimension
#   gold_dim_location       → City / State dimension
#   gold_dim_fulfillment    → Fulfillment / Shipping dimension
#   gold_dim_date           → Date / Time dimension
#   gold_dim_promotion      → Promotion type dimension
#   gold_dim_customer       → Customer type dimension
# """)


StatementMeta(, c087b22b-daa1-4920-aa8b-b56c953d9256, 16, Finished, Available, Finished, False)


GOLD LAYER VERIFICATION
  ecommerce_gold.gold.gold_fact_sales → 128748 rows
  ecommerce_gold.gold.gold_dim_product →   7190 rows
  ecommerce_gold.gold.gold_dim_location →  13540 rows
  ecommerce_gold.gold.gold_dim_fulfillment →      3 rows
  ecommerce_gold.gold.gold_dim_date   →     91 rows
  ecommerce_gold.gold.gold_dim_promotion →      5 rows
  ecommerce_gold.gold.gold_dim_customer →      2 rows

--- Join Verification ---
Joined rows: 128748
Expected   : 128748
All joins verified — no orphan keys!


What This Does
This tests all 4 relationships of your star schema to make sure every fact row can find its matching dimension row.
Why left Join is Used Here
python# left join = keep ALL fact rows
 even if no matching dim row is found
.join(dim_p, "product_key", "left")

 This is intentional for testing purposes:
 If a fact row has no matching dim row
 → it still appears in joined result
 → but dim columns will be NULL
 → row count stays the same
 → we can detect orphan keys
```

### What Orphan Keys Are
```
Orphan key = a foreign key in fact that has
             NO matching row in the dimension table

Example:
fact_sales has product_key = "abc123"
dim_product has NO row with product_key = "abc123"
→ this fact row is an orphan ❌
→ in Power BI this order would show blank product info
How the Check Works
python# Scenario 1 — All joins work perfectly
joined.count() = 128,000   ← same as fact
fact.count()   = 128,000
→ ✅ All joins verified — no orphan keys!

 Scenario 2 — Some keys don't match
 With left join, row count stays same
 but NULL values appear in dim columns
 Additional check needed for NULLs:
joined.filter(col("sku").isNull()).count()
→ tells you exactly how many orphan keys exist
